# CommitCraft: QLoRA training

Settings: GPU T4 x2, Internet on. Add-ons > Secrets: attach `HF_TOKEN` and `WANDB_API_KEY`.

In [ ]:
# Install the same library versions that were tested locally.
!pip install -q transformers==5.22.1 peft==0.21.2 trl==1.17.1 bitsandbytes==0.50.2 accelerate==1.15.0 datasets wandb rich python-dotenv

In [ ]:
# Start from a fresh copy, so running this cell again never nests or reuses old code.
%cd /kaggle/working
!rm -rf CommitCraft
!git clone --depth 1 https://github.com/shivankarpatra04/CommitCraft.git
%cd CommitCraft

In [ ]:
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_USERNAME"] = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
# A 4-bit 1.5B model fits on one GPU, so training uses only the first one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Ready for", os.environ["HF_USERNAME"])

## Smoke test

20 steps, nothing uploaded to the Hub. Run this first and check that it finishes without errors.

In [ ]:
!python -m train.train --max-steps 20
# A failed ! command does not stop a notebook, so check its exit code.
assert _exit_code == 0, "Training failed, so the notebook stops here."  # noqa: F821

## Full run

2 epochs. **This cell makes the adapter public on the Hugging Face Hub at the end.** Copy the W&B run URL and the final train and eval loss from the output.

In [ ]:
!rm -rf outputs/checkpoints
!python -m train.train --push
assert _exit_code == 0, "Training failed, so the notebook stops here."  # noqa: F821

## Evaluation

Run the install, clone and secrets cells first, then these two cells by hand. **Skip the smoke test and Full run cells, and do not use Save Version**, because that trains again. Also attach the `ANTHROPIC_API_KEY` secret.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
!pip install -q evaluate==0.4.6 rouge_score==0.1.2 anthropic==2.6.0
!python -m eval.evaluate --limit 5
assert _exit_code == 0, "Evaluation failed, so the notebook stops here."  # noqa: F821

In [ ]:
!python -m eval.evaluate
assert _exit_code == 0, "Evaluation failed, so the notebook stops here."  # noqa: F821
!cat outputs/results.md